# 02 — Descriptors and invariance (M2)

**Goal:** turn each molecule's atoms and coordinates into a fixed-length vector a model can use, and prove that the
vector doesn't change when the molecule is rotated, moved, or has its atoms listed in a different order.

M2 acceptance criteria (PLAN §9.4):
- composition and Coulomb-matrix-spectrum descriptors implemented (`src/qm9dipole/descriptors.py`);
- invariance tests pass, and the negative control fails (`tests/test_invariance.py`).

Run with the project environment as the kernel (README.md, "Setup"); every cell is safe to re-run.

In [ ]:
# Environment check: stops here, with setup instructions, unless this kernel is the project
# environment (README.md, "Setup"). Version differences from requirements.txt only warn.
try:
    from qm9dipole.provenance import check_environment
except ModuleNotFoundError as e:
    raise RuntimeError(
        "The qm9dipole package is not installed in this kernel's environment. Follow "
        "README.md (Setup), then select that environment as the notebook kernel."
    ) from e

print(check_environment())

## 1. Three descriptors

| Descriptor | Length | What it encodes | Role |
|---|---|---|---|
| `composition` | 5 | counts of C, H, N, O, F | headline input (composition only); 5 qubits in M4 |
| `cm_spectrum` | 29 | eigenvalues of the **Coulomb matrix**, sorted by size, zero-padded | headline input (composition + geometry); compressed to 8 numbers = 8 qubits in M4 |
| `raw_coordinates` | 87 | x, y, z of each atom, flattened | **negative control** only: deliberately not invariant |

The Coulomb matrix (Rupp et al. 2012) has one row and column per atom. The diagonal entry 0.5·Z^2.4 depends only on
the element. The off-diagonal entry Z_i·Z_j / distance is the repulsion between two nuclei, so it encodes the
geometry. Distances don't change when a molecule rotates or moves, and a matrix's eigenvalues don't depend on the
order of its rows, so the sorted spectrum is invariant by construction. The tests below check that the code
delivers on the math.

In [ ]:
import numpy as np
import pandas as pd

from qm9dipole.data import load_qm9_table
from qm9dipole.descriptors import DESCRIPTORS, MODEL_DESCRIPTORS
from qm9dipole.splits import load_splits, training_pool

table = load_qm9_table()
splits = load_splits()
P = table[table["id"].isin(training_pool(splits, table))].reset_index(drop=True)
print(f"training pool P: {len(P):,} molecules (all analyses below stay inside P; no test molecules)")

np.set_printoptions(precision=2, suppress=True, linewidth=110)
for mol_id, name in [(1, "methane CH4"), (3, "water H2O")]:
    m = table.loc[table["id"] == mol_id].iloc[0]
    print(f"\n{name}")
    for d, fn in DESCRIPTORS.items():
        v = fn(m["Z"], m["R"])
        print(f"  {d:16s} length {len(v):3d}: {v[:8]}{' ...' if len(v) > 8 else ''}")

## 2. Invariance checks (PLAN §8)

For 100 random training-pool molecules, each descriptor is computed before and after a random rotation, a random
translation (up to 10 Å) and a random relabeling of the atoms. The table reports how much the descriptor changed.
"Invariant" means the largest change over the 100 molecules is at most 1e-8, which is far above floating-point noise.

The negative control must **fail**. If it passed, these checks couldn't detect a broken descriptor.

In [ ]:
from qm9dipole.invariance import ATOL, invariance_table
from qm9dipole.provenance import save_result

SEED = 2026
sample = P.sample(100, random_state=SEED)
invariance = invariance_table(list(zip(sample["Z"], sample["R"])), seed=SEED)
save_result(invariance, "m2_descriptor_invariance", seed=SEED, n_molecules=len(sample), atol=ATOL,
            sample="100 random molecules from the training pool")

expected = invariance["descriptor"].isin(MODEL_DESCRIPTORS)
assert (invariance["invariant"] == expected).all(), "unexpected invariance pattern"
print("model descriptors invariant under all three transforms; negative control fails all three")
invariance.style.format({"mean_change": "{:.1e}", "max_change": "{:.1e}"}).hide(axis="index")

Composition counts can't change at all (exactly 0). The Coulomb spectrum changes by about 1e-13, which is
rounding error in the eigenvalue solver. Raw coordinates change by ångströms. This table is saved to
`results/m2_descriptor_invariance.csv` (with a `.meta.json` provenance sidecar) for the writeup.

## 3. What composition can't see

Composition is invariant for free, but it gives every isomer (same formula, different structure) the same
vector, so a composition-only model must predict the same |μ| for all of them. How much does that cost?

To bound it, predict each training-pool molecule by the **median |μ| of its own formula**. That is the best any
composition-only model could do on these molecules, even with perfect memory of them.

In [ ]:
per_formula_median = P.groupby("formula")["mu"].transform("median")
floor = (P["mu"] - per_formula_median).abs().mean()
global_mae = (P["mu"] - P["mu"].median()).abs().mean()
print(f"MAE predicting one global median        : {global_mae:.2f} D")
print(f"MAE predicting each formula's own median: {floor:.2f} D  <- floor for any composition-only model")
print(f"knowing the formula cuts the error by at most {1 - floor / global_mae:.0%}; geometry has to supply the rest")

sizes = P.groupby("formula").size()
example = (P[P["formula"].isin(sizes[sizes >= 3].index)]
           .sort_values(["n_heavy", "formula"]).iloc[0]["formula"])
print(f"\nexample: the isomers of {example} in the training pool")
P.loc[P["formula"] == example, ["id", "formula", "smiles", "mu"]].sort_values("mu").reset_index(drop=True)

## M2 status

If every cell ran, M2's acceptance criteria are met: both model descriptors are invariant to rotation,
translation and atom relabeling, and the raw-coordinate negative control fails all three checks, in the unit tests
(`tests/test_invariance.py`, synthetic and real molecules) and in the table above.

**Next (M3):** classical baselines (mean, RBF kernel ridge, random forest) with learning curves from
cross-validation only. The 8-number "compressed" spectrum is a PCA fit inside each training set, so it's built in M3.
**Before M3:** settle the open anchor-skew decision in `docs/DECISIONS.md`.